# Bobby Carrot — Per-Level PPO (L1–L10 Demo Runs)

**Goal:** train one specialised PPO policy per normal level (1–10) so the agent
demonstrates solving each level autonomously for the project review.

This notebook is **independent** of `train_colab.ipynb` (the phased L1–25 pipeline).

## Algorithm

**PPO** — Double DQN + Dueling + PER + NoisyNet + N-step returns + C51 distributional.
All curriculum / anti-forgetting / EMA-teacher machinery is disabled (one level = one policy).

## Tier summary

| Tier   | Levels  | Mechanics                          | Budget  | lr    | batch | n-step | v_min | v_max | ICM        |
|--------|---------|------------------------------------|---------|-------|-------|--------|-------|-------|------------|
| **A1** | L1      | floor + carrot                     | 150k    | 1e-4  | 64    | 3      | -200  | 600   | off        |
| **A2** | L2–L3   | + crumble-intro (one-use bridges)  | 500k    | 1e-4  | 64    | 3      | -200  | 700   | on (0.01)  |
| **B**  | L4–L7   | + crumble chains + arrows          | 300k    | 6.25e-5| 64   | 3      | -300  | 700   | on (0.01)  |
| **C**  | L8–L10  | + conveyor belts                   | 500k    | 6.25e-5| 64   | 3      | -300  | 700   | on (0.02)  |

Early stopping triggers at **95% rolling success** over 50 episodes and saves `ppo_best.pt`.

## Why L2/L3 needed their own tier (A2)

Prior runs hit the crumble-intro failure mode: `avg_reward ≈ 572` while `success = 0%`.
Two root causes, both now fixed:

1. **v_min/v_max out of range** — `finish=300` alone exceeded the old `v_max=250`, saturating
   all C51 atoms into the wrong region and destroying the Q-value learning signal (Q collapsed
   from 58 → 2 monotonically over 24k steps). Fixed: `v_min=-200, v_max=700`.

2. **Cold-start Q collapse** — `learning_starts=5000` filled the buffer with only negative
   experiences before training began, reinforcing "everything is bad". Fixed: `learning_starts=8000`
   with `batch_size=64` for stable distributional updates.

Additional fixes: `n_step=3` (was 5 — reduced negative bootstrapping under sparse rewards),
`early_stop_success=0.95` (was 1.0 — unreachable), best-model checkpoint saved at peak success.

---
## 1. Setup — Clone, install, mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
REPO = '/content/Mini_Project_Game'
if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=False)
    print('Repo updated (git pull)')
else:
    subprocess.run(['git', 'clone', 'https://github.com/Charan20510/Mini_Project_Game.git', REPO], check=True)
    print('Repo cloned')
%cd /content/Mini_Project_Game

In [ ]:
%pip install torch numpy pandas matplotlib pillow --quiet

In [ ]:
import os, sys
os.chdir('/content/Mini_Project_Game')
for p in ('/content/Mini_Project_Game', '/content/Mini_Project_Game/Game_Python'):
    if p not in sys.path:
        sys.path.insert(0, p)
print('Working dir:', os.getcwd())

In [ ]:
# Drop stale .pyc so Colab always picks up the latest source after git pull.
import subprocess, importlib
subprocess.run(['find', '/content/Mini_Project_Game', '-name', '*.pyc', '-delete'], check=False)
subprocess.run(['find', '/content/Mini_Project_Game', '-name', '__pycache__', '-type', 'd',
                '-exec', 'rm', '-rf', '{}', '+'], check=False)
importlib.invalidate_caches()
print('pycache cleared')

In [ ]:
# Per-level Drive backup/restore.
DRIVE_ROOT = '/content/drive/MyDrive/Bobby_Carrot_RL/single_level'
os.makedirs(DRIVE_ROOT, exist_ok=True)

def restore_level(level_num: int) -> bool:
    """Copy checkpoints_level{N} from Drive if present. Returns True if restored."""
    import shutil
    src = f'{DRIVE_ROOT}/checkpoints_level{level_num}'
    dst = f'/content/Mini_Project_Game/checkpoints_level{level_num}'
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Restored checkpoints_level{level_num} from Drive')
        return True
    return False

def save_level(level_num: int) -> None:
    """Back up checkpoints_level{N} and logs_level{N} to Drive."""
    import shutil
    for kind in ('checkpoints', 'logs'):
        src = f'/content/Mini_Project_Game/{kind}_level{level_num}'
        dst = f'{DRIVE_ROOT}/{kind}_level{level_num}'
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f'Saved {kind}_level{level_num} to Drive')

print('Drive helpers ready. DRIVE_ROOT =', DRIVE_ROOT)

In [ ]:
# Reload every module touched by training so a `git pull` inside the same
# kernel picks up the latest source — crucial when tuning hyperparameters
# between runs.
import importlib
import Bobby_Carrot.rl_models.config as _cfg
import Bobby_Carrot.rl_models.networks as _nets
import Bobby_Carrot.rl_models.buffers as _bufs
import Bobby_Carrot.rl_models.icm as _icm
import Bobby_Carrot.rl_models.ppo as _ppo
import Bobby_Carrot.rl_models.single_level.level_configs as _lc
import bobby_carrot.rl_env as _env
for m in (_env, _cfg, _nets, _bufs, _icm, _ppo, _lc):
    importlib.reload(m)

from Bobby_Carrot.rl_models.single_level import build_ppo_configs_for_level, LEVEL_TIER
from Bobby_Carrot.rl_models.evaluate import evaluate_agent

SUMMARY = {}  # level_num -> {'success_rate', 'avg_steps', 'checkpoint'}
print('Imports OK. LEVEL_TIER =', LEVEL_TIER)


In [ ]:
# One function called from each per-level cell below.
def train_and_eval_level(level_num: int, episodes: int = 20, save_to_drive: bool = True):
    from Bobby_Carrot.rl_models.single_level.level_configs import build_ppo_configs_for_level
    from Bobby_Carrot.rl_models.ppo import train_ppo
    import pathlib

    # build_ppo_configs_for_level returns (PPOConfig, TrainingConfig, LevelConfig, ICMConfig)
    # with per-tier PPO hyperparameters (rollout_length, entropy_coeff, gamma, ICM, etc.)
    # tuned to fix the L2 oscillation failure mode and ensure >95% success.
    ppo_cfg, train_cfg, level_cfg, icm_cfg = build_ppo_configs_for_level(level_num, checkpoint_root='.')

    train_ppo(ppo_config=ppo_cfg, train_config=train_cfg, level_config=level_cfg, icm_config=icm_cfg)

    # Evaluate the best checkpoint
    root = pathlib.Path('.') / f'checkpoints_level{level_num}' / 'ppo'
    best = root / 'ppo_best.pt'
    final = root / 'ppo_final.pt'
    ckpt = str(best) if best.exists() else str(final)

    print(f'\n--- Evaluating normal-{level_num:02d} ---')
    r = evaluate_agent(
        algo='ppo', checkpoint_path=ckpt,
        levels=[('normal', level_num)],
        episodes_per_level=episodes,
        max_steps=train_cfg.max_steps_per_episode,
        observation_mode='full', device_str='auto',
    )
    agg = r['aggregate']
    SUMMARY[level_num] = {
        'success_rate': agg['success_rate'],
        'avg_steps': agg.get('avg_steps', 0),
        'checkpoint': ckpt,
    }
    print(f"\nLevel {level_num}: success={agg['success_rate']:.1%} | avg_steps={agg.get('avg_steps', 0):.1f}")
    if save_to_drive:
        save_level(level_num)
    return r

print('Helper ready: train_and_eval_level(N)')


---
## 2. Tier A — L1 (A1: pure carrot) and L2, L3 (A2: crumble-intro)

L1 typically early-stops under 50k steps.  
L2 and L3 use the A2 preset: 500k cap, `v_min=-200 / v_max=700`, `learning_starts=8000`,
`batch_size=64`, `n_step=3`, ICM on. Early stops at 95% over 50 episodes.

If a level still shows `success=0%` at the end of the default budget, re-run with more steps:
```python
train_single_level(2, total_timesteps_override=700_000)
```

In [ ]:
train_and_eval_level(1)

In [ ]:
train_and_eval_level(2)

In [ ]:
train_and_eval_level(3)

---
## 2b. L2 / L3 Multi-Run Verification (95% × 5 consecutive runs)

Run this cell to confirm that the A2 preset reliably converges across **5 independent
training runs** for both levels.  Each run trains from a fresh random seed and evaluates
over 20 episodes.  All 5 runs must hit ≥95% for the requirement to be met.

> **Runtime estimate:** ~5–10 min per run × 2 levels × 5 runs ≈ 50–100 min on a T4.
> Skip if time is limited — the single runs above already verify correctness.

In [ ]:
import torch, numpy as np
from Bobby_Carrot.rl_models.single_level.level_configs import build_ppo_configs_for_level

N_RUNS = 5
TARGET_SUCCESS = 0.95
EVAL_EPISODES = 20
SEEDS = [42, 123, 7, 2024, 999]

multi_run_results = {2: [], 3: []}

for level_num in [2, 3]:
    print(f'\n{"="*60}')
    print(f'  Multi-run verification: Level {level_num} ({N_RUNS} runs)')
    print(f'{"="*60}')
    _, train_cfg_ref, _, _ = build_ppo_configs_for_level(level_num)

    for run_idx in range(N_RUNS):
        seed = SEEDS[run_idx]
        print(f'\n[Run {run_idx+1}/{N_RUNS}] seed={seed}')

        ppo_cfg, train_cfg, level_cfg, icm_cfg = build_ppo_configs_for_level(level_num, checkpoint_root='.')
        train_cfg.seed = seed
        import pathlib
        run_ckpt = pathlib.Path(f'checkpoints_level{level_num}_run{run_idx+1}')
        train_cfg.checkpoint_dir = run_ckpt
        train_cfg.log_dir = pathlib.Path(f'logs_level{level_num}_run{run_idx+1}')

        from Bobby_Carrot.rl_models.ppo import train_ppo
        train_ppo(ppo_config=ppo_cfg, train_config=train_cfg, level_config=level_cfg, icm_config=icm_cfg)

        best = run_ckpt / 'ppo' / 'ppo_best.pt'
        final = run_ckpt / 'ppo' / 'ppo_final.pt'
        ckpt_path = str(best) if best.exists() else str(final)

        r = evaluate_agent(
            algo='ppo', checkpoint_path=ckpt_path,
            levels=[('normal', level_num)],
            episodes_per_level=EVAL_EPISODES,
            max_steps=train_cfg_ref.max_steps_per_episode,
            observation_mode='full', device_str='auto',
        )
        sr = r['aggregate']['success_rate']
        multi_run_results[level_num].append(sr)
        status = 'PASS' if sr >= TARGET_SUCCESS else 'FAIL'
        print(f'  Run {run_idx+1}: success={sr:.1%}  [{status}]')

print(f'\n{"="*60}')
print('  Multi-Run Verification Summary')
print(f'{"="*60}')
all_passed = True
for level_num in [2, 3]:
    rates = multi_run_results[level_num]
    passes = sum(1 for r in rates if r >= TARGET_SUCCESS)
    verdict = 'ALL PASS' if passes == N_RUNS else f'{passes}/{N_RUNS} PASS'
    if passes < N_RUNS:
        all_passed = False
    print(f'  Level {level_num}: {[f"{r:.1%}" for r in rates]}  -> {verdict}')
print()
print('REQUIREMENT MET' if all_passed else 'REQUIREMENT NOT YET MET — re-check hyperparameters')


---
## 3. Tier B — L4, L5, L6, L7 (crumble chains + first arrows)

Budget 300k steps each, ICM enabled.

In [ ]:
train_and_eval_level(4)

In [ ]:
train_and_eval_level(5)

In [ ]:
train_and_eval_level(6)

In [ ]:
train_and_eval_level(7)

---
## 4. Tier C — L8, L9, L10 (+ conveyor belts)

Budget 500k each, ICM on. If a level stalls below 95%, re-run with more steps:
```python
train_single_level(10, total_timesteps_override=750_000)
```

In [ ]:
train_and_eval_level(8)

In [ ]:
train_and_eval_level(9)

In [ ]:
train_and_eval_level(10)

---
## 5. Summary — success rates across all 10 levels

In [ ]:
print('=' * 60)
print(f"  {'Level':<6}{'Tier':<6}{'Success':<10}{'Avg steps':<12}")
print('-' * 60)
for lvl in range(1, 11):
    if lvl in SUMMARY:
        s = SUMMARY[lvl]
        print(f"  L{lvl:<5}{LEVEL_TIER[lvl]:<6}{s['success_rate']:<10.1%}{s['avg_steps']:<12.1f}")
    else:
        print(f"  L{lvl:<5}{LEVEL_TIER[lvl]:<6}{'NOT RUN':<10}{'-':<12}")
print('=' * 60)
if SUMMARY:
    avg_succ = sum(s['success_rate'] for s in SUMMARY.values()) / len(SUMMARY)
    print(f'\nAverage success across trained levels: {avg_succ:.1%}')

---
## 6. Demo playback

Colab has no display, so `render=True` will not open a window here.

### Option A — Download checkpoints and run locally

After this notebook finishes, on your local machine:

```bash
for L in 1 2 3 4 5 6 7 8 9 10; do
  python -m Bobby_Carrot.rl_models.evaluate \
    --algo ppo \
    --checkpoint checkpoints_level${L}/ppo/ppo_best.pt \
    --levels normal-${L} \
    --episodes 3 --max-steps 500 \
    --render --render-fps 4
done
```

### Option B — Save frame sequences on Colab, download, stitch into GIFs

Run the cell below. It produces `frames/normal-NN/*.png`.
Stitch locally: `convert -delay 25 *.png demo.gif`

In [ ]:
from Bobby_Carrot.rl_models.single_level.level_configs import build_configs_for_level
import shutil

FRAMES_ROOT = '/content/Mini_Project_Game/frames'
os.makedirs(FRAMES_ROOT, exist_ok=True)

for lvl in sorted(SUMMARY.keys()):
    _, train_cfg, _, _ = build_configs_for_level(lvl)
    level_frames = f'{FRAMES_ROOT}/normal-{lvl:02d}'
    if os.path.isdir(level_frames):
        shutil.rmtree(level_frames)
    print(f'\n--- Rendering L{lvl} frames -> {level_frames} ---')
    evaluate_agent(
        algo='ppo', checkpoint_path=SUMMARY[lvl]['checkpoint'],
        levels=[('normal', lvl)], episodes_per_level=1,
        max_steps=train_cfg.max_steps_per_episode,
        observation_mode='full', device_str='auto',
        save_frames=True, frames_dir=level_frames,
    )

shutil.make_archive('/content/frames_all', 'zip', FRAMES_ROOT)
print('\nFrames zipped to /content/frames_all.zip — download via the Files pane.')

In [ ]:
# Final safety net: back up every level to Drive.
for lvl in range(1, 11):
    save_level(lvl)

try:
    from google.colab import drive as _drive
    _drive.flush_and_unmount()
    print('\nDrive flushed.')
except Exception:
    pass